!/usr/bin/env python
coding: utf-8

LLM Bias Evaluation Pipeline — Category-Level Analysis
**Author:** Mohanraj Ramanujam | PA2312049010014 | M.Tech AI, SRM University
**Guide:** Dr. S. Godfrey Winster

Pipeline Overview
```
[1] Validate CSVs          → Check column names and data quality
[2] Translate Datasets     → English → Tamil (IndicTrans2) + Hindi (NLLB)
[3] Stereotype Evaluation  → Log-likelihood scoring per category per model per language
[4] Toxicity Evaluation    → Text generation + Detoxify subtype scoring
[5] Visualise Results      → Generate all 8+ charts
[6] Summary Report         → Print all scores in table format
```

**Checkpointing:** Each step saves progress. If the run crashes, re-run from the
same cell — completed work will be detected and skipped automatically.

**Estimated runtime on RTX 4090:** ~6–8 hours for all 5 models × 3 languages.


In [1]:
import os, sys, json
import pandas as pd

# Add pipeline root to path
PIPELINE_ROOT = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, PIPELINE_ROOT)

from modules.config import (
    OUTPUT_DIR, CHECKPOINT_DIR, MODEL_CONFIGS, LANGUAGES,
    STEREOTYPE_CSV, TOXICITY_CSV,
)
from modules.validate_data   import run_validation
from modules.translate       import translate_datasets
from modules.evaluate_stereotype import run_stereotype_evaluation
from modules.evaluate_toxicity   import run_toxicity_evaluation
from modules.visualize           import generate_all_charts

print("Pipeline root:", PIPELINE_ROOT)
print("Output dir:   ", OUTPUT_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print()
print("Models to evaluate:")
for name in MODEL_CONFIGS:
    cfg = MODEL_CONFIGS[name]
    print(f"  {name:15s}  {cfg['repo']}")


Pipeline root: /workspace/llm-bias-evaluation
Output dir:    /workspace/llm-bias-evaluation/outputs
Checkpoint dir: /workspace/llm-bias-evaluation/checkpoints

Models to evaluate:
  LLaMA-2-7B       NousResearch/Llama-2-7b-hf
  BLOOM-560M       bigscience/bloom-560m
  Falcon-1B        tiiuae/falcon-rw-1b
  Mistral-7B       mistralai/Mistral-7B-v0.1
  GPT-J-6B         EleutherAI/gpt-j-6b


In [4]:
!python3 -c "import torch; print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0))"

CUDA: True
GPU: NVIDIA GeForce RTX 4090


## STEP 1 — Validate input CSVs
Place your files in:
pipeline/data/stereo_type_dataset.csv
pipeline/data/toxicity.csv

Required columns:
stereo_type_dataset.csv → sent_more, sent_less, bias_type
toxicity.csv            → prompt

If your columns have different names, update config.py:
STEREO_COL_MORE, STEREO_COL_LESS, STEREO_COL_CAT, TOXICITY_COL_PROMPT


In [2]:

stereo_ok, tox_ok, stereo_df, tox_df = run_validation()

assert stereo_ok, "Fix stereotype CSV errors above before continuing."
assert tox_ok,    "Fix toxicity CSV errors above before continuing."

print("\n✓ Both CSVs validated. Proceeding to translation.")


 Validating stereo_type_dataset.csv
  Rows: 1,508
  Columns found: ['sent_more', 'sent_less', 'stereo_antistereo', 'bias_type', 'annotations', 'anon_writer', 'anon_annotators']
  OK  : Column 'sent_more' found (stereotypical sentence)
  OK  : Column 'sent_less' found (anti-stereotypical sentence)
  OK  : Column 'bias_type' found (bias category label)

  Bias categories found (9):
    ✓ age: 87 pairs
    ✓ disability: 60 pairs
    ✓ gender: 262 pairs
    ✓ nationality: 159 pairs
    ✓ physical-appearance: 63 pairs
    ✓ race-color: 516 pairs
    ✓ religion: 105 pairs
    ✓ sexual-orientation: 84 pairs
    ✓ socioeconomic: 172 pairs

  Sample rows:
                                                                                                                                             sent_more                                                                                                                                              sent_less     bias_type
He couldn't figure out the is

## STEP 2 — Translate datasets to Tamil and Hindi
Outputs (saved to outputs/):
stereo_English.csv, stereo_Tamil.csv, stereo_Hindi.csv
toxicity_English.csv, toxicity_Tamil.csv, toxicity_Hindi.csv

This step is SKIPPED automatically if output files already exist.
Estimated time: ~45–90 minutes depending on dataset size.


In [3]:

translate_datasets(stereo_df, tox_df)
print("\n✓ Translation complete.")


2026-04-02 17:31:38,269 [translate] English CSVs saved.
2026-04-02 17:31:38,270 [translate] Tamil CSVs already exist — skipping.
2026-04-02 17:31:38,271 [translate] Hindi CSVs already exist — skipping.
2026-04-02 17:31:38,271 [translate] All translations complete.



✓ Translation complete.


## STEP 3 — Stereotype evaluation (all models, all languages)
For each model × language:
- Computes log P(sent_more) and log P(sent_less) for every pair
- Labels pair as stereotypical if log P(sent_more) > log P(sent_less)
- Computes CSBS per category and overall SBS

Checkpoints saved per model × language to checkpoints/stereo_*.csv
Final outputs:
outputs/stereo_raw_predictions.csv
outputs/stereotype_scores.json
outputs/stereotype_scores.csv

To run only specific models, change model_names list, e.g.:
model_names = ["LLaMA-2-7B", "Mistral-7B"]
To run only specific languages:
languages = ["English"]


In [5]:

model_names = list(MODEL_CONFIGS.keys())   # all 5 models
languages   = LANGUAGES                    # English, Tamil, Hindi

stereo_scores, stereo_scores_df = run_stereotype_evaluation(
    model_names=model_names,
    languages=languages,
)

print("\n✓ Stereotype evaluation complete.")
print("\nOverall SBS summary:")
print(stereo_scores_df[["model", "language", "overall_sbs"]].to_string(index=False))


2026-04-02 11:26:37,392 [translate] Loading LLaMA-2-7B (NousResearch/Llama-2-7b-hf)...
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-04-02 11:26:38,813 [translate] We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-04-02 11:26:41,725 [translate]   LLaMA-2-7B loaded.
2026-04-02 11:26:41,726 [translate]   Checkpoint found: LLaMA-2-7B/English — skipping.
2026-04-02 11:26:41,733 [translate]   Checkpoint found: LLaMA-2-7B/Tamil — skipping.
2026-04-02 11:26:41,751 [translate]   Checkpoint found: LLaMA-2-7B/Hindi — skipping.
2026-04-02 11:26:41,876 [translate]   LLaMA-2-7B unloaded and GPU memory cleared.
2026-04-02 11:26:41,877 [translate] Loading BLOOM-560M (bigscience/bloom-560m)...
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, 

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Exception: data did not match any variant of untagged enum PyPreTokenizerTypeWrapper at line 40 column 3

## STEP 4 — Toxicity evaluation (all models, all languages)
For each model × language:
- Generates 50-token greedy continuation per prompt
- Scores continuation with Detoxify (6 subtypes)
- Computes STBS (mean per subtype)

Checkpoints saved to:
checkpoints/toxicity_cont_<model>_<lang>.csv   (continuations)
checkpoints/toxicity_scored_<model>_<lang>.csv  (with subtype scores)
Final outputs:
outputs/toxicity_raw_scored.csv
outputs/toxicity_scores.json
outputs/toxicity_scores.csv


In [ ]:
model_names = list(MODEL_CONFIGS.keys())   # all 5 models
model_names = ["LLaMA-2-7B", "BLOOM-560M", "Mistral-7B", "GPT-J-6B"]
languages   = LANGUAGES   

tox_scores, tox_scores_df = run_toxicity_evaluation(
    model_names=model_names,
    languages=languages,
)

print("\n✓ Toxicity evaluation complete.")
print("\nTBS summary (Toxicity subtype):")
tbs_summary = tox_scores_df[["model", "language", "stbs_toxicity"]].copy()
print(tbs_summary.to_string(index=False))


2026-04-02 17:32:04,277 [translate] Loading LLaMA-2-7B (NousResearch/Llama-2-7b-hf)...
2026-04-02 17:32:04,591 [translate] HTTP Request: HEAD https://huggingface.co/NousResearch/Llama-2-7b-hf/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 17:32:04,880 [translate] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Llama-2-7b-hf/8efe6c9b93655b934e27bd9981e3ec13e55aee9d/config.json "HTTP/1.1 200 OK"
2026-04-02 17:32:05,714 [translate] HTTP Request: HEAD https://huggingface.co/NousResearch/Llama-2-7b-hf/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 17:32:05,746 [translate] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Llama-2-7b-hf/8efe6c9b93655b934e27bd9981e3ec13e55aee9d/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-02 17:32:05,874 [translate] HTTP Request: GET https://huggingface.co/api/models/NousResearch/Llama-2-7b-hf/tree/main/additional_chat_templates?recursive=fa

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-04-02 17:32:10,135 [translate] HTTP Request: HEAD https://huggingface.co/NousResearch/Llama-2-7b-hf/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 17:32:10,171 [translate] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Llama-2-7b-hf/8efe6c9b93655b934e27bd9981e3ec13e55aee9d/generation_config.json "HTTP/1.1 200 OK"
2026-04-02 17:32:10,175 [translate]   LLaMA-2-7B loaded.
2026-04-02 17:32:10,176 [translate]   Checkpoint found: LLaMA-2-7B/English — skipping generation.
2026-04-02 17:32:10,271 [translate]   Checkpoint found: LLaMA-2-7B/Tamil — skipping generation.
2026-04-02 17:32:10,348 [translate]   Checkpoint found: LLaMA-2-7B/Hindi — skipping generation.
2026-04-02 17:32:10,582 [translate]   LLaMA-2-7B unloaded and GPU memory cleared.
2026-04-02 17:32:10,583 [translate] Loading BLOOM-560M (bigscience/bloom-560m)...
2026-04-02 17:32:10,718 [translate] HTTP Request: HEAD https://huggingface.co/bigscience/bloom-560m/re

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

2026-04-02 17:32:14,817 [translate] HTTP Request: HEAD https://huggingface.co/bigscience/bloom-560m/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-04-02 17:32:14,934 [translate] HTTP Request: HEAD https://huggingface.co/bigscience/bloom-560m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 17:32:14,975 [translate] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bigscience/bloom-560m/ac2ae5fab2ce3f9f40dc79b5ca9f637430d24971/config.json "HTTP/1.1 200 OK"
2026-04-02 17:32:14,979 [translate]   BLOOM-560M loaded.
2026-04-02 17:32:14,997 [translate]   Checkpoint found: BLOOM-560M/English — skipping generation.
2026-04-02 17:32:15,081 [translate]   Generating continuations: BLOOM-560M/Tamil...
  BLOOM-560M/Tamil — generating: 100%|██████████| 349/349 [04:49<00:00,  1.20it/s]
2026-04-02 17:37:04,942 [translate]   Continuations saved: /workspace/llm-bias-evaluation/checkpoints/toxicity_cont_BLOOM-560M_Tamil.csv
2026-04-02 17:37:04,942 [

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: None
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-02 17:37:06,185 [translate] HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-02 17:37:06,303 [translate] HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-04-02 17:37:06,422 [translate] HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-02 17:37:06,771 [translate] HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree

# STEP 4.1 Generate score for Falcon model Alone. Overrides the existing scores

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from modules.evaluate_stereotype import run_stereotype_evaluation
from modules.evaluate_toxicity import run_toxicity_evaluation
from modules.config import LANGUAGES

# ── Falcon stereotype ─────────────────────────────────────────────────────────
print("Running Falcon-1B stereotype evaluation...")
run_stereotype_evaluation(
    model_names=["Falcon-1B"],
    languages=LANGUAGES,
)
print("✓ Falcon stereotype done.")

# ── Falcon toxicity ───────────────────────────────────────────────────────────
print("Running Falcon-1B toxicity evaluation...")
run_toxicity_evaluation(
    model_names=["Falcon-1B"],
    languages=LANGUAGES,
)
print("✓ Falcon toxicity done.")

# STEP 4.2 Generate scores for all 5 models

In [ ]:
import pandas as pd, glob, json, os
from modules.evaluate_stereotype import compute_bias_scores
from modules.evaluate_toxicity import compute_stbs
from modules.config import OUTPUT_DIR, CHECKPOINT_DIR

# ── Rebuild stereotype scores (all 5 models) ──────────────────────────────────
print("Rebuilding stereotype scores...")
all_stereo = pd.concat(
    [pd.read_csv(f) for f in glob.glob(os.path.join(CHECKPOINT_DIR, "stereo_*.csv"))],
    ignore_index=True
)
stereo_scores = compute_bias_scores(all_stereo)

with open(os.path.join(OUTPUT_DIR, "stereotype_scores.json"), "w") as f:
    json.dump(stereo_scores, f, indent=2)

rows = []
for model, langs_dict in stereo_scores.items():
    for language, data in langs_dict.items():
        row = {"model": model, "language": language,
               "overall_sbs": data["overall_sbs"],
               "n_pairs": data["n_pairs"]}
        for cat, val in data["categories"].items():
            row[f"csbs_{cat}"] = val
        rows.append(row)
pd.DataFrame(rows).to_csv(
    os.path.join(OUTPUT_DIR, "stereotype_scores.csv"), index=False)
print(f"  ✓ stereotype_scores.csv updated — {len(rows)} rows ({len(stereo_scores)} models)")

# ── Rebuild toxicity scores (all 5 models) ────────────────────────────────────
print("Rebuilding toxicity scores...")
all_tox = pd.concat(
    [pd.read_csv(f) for f in glob.glob(os.path.join(CHECKPOINT_DIR, "toxicity_scored_*.csv"))],
    ignore_index=True
)
tox_scores = compute_stbs(all_tox)

with open(os.path.join(OUTPUT_DIR, "toxicity_scores.json"), "w") as f:
    json.dump(tox_scores, f, indent=2)

rows = []
for model, langs_dict in tox_scores.items():
    for language, data in langs_dict.items():
        row = {"model": model, "language": language,
               "n_prompts": data["n_prompts"]}
        for sub, val in data["subtypes"].items():
            row[f"stbs_{sub}"] = val
        rows.append(row)
pd.DataFrame(rows).to_csv(
    os.path.join(OUTPUT_DIR, "toxicity_scores.csv"), index=False)
print(f"  ✓ toxicity_scores.csv updated — {len(rows)} rows ({len(tox_scores)} models)")

# ── Verify all 5 models present ───────────────────────────────────────────────
print("\n── Models in stereotype_scores.csv ──")
sdf = pd.read_csv(os.path.join(OUTPUT_DIR, "stereotype_scores.csv"))
print(sdf[["model", "language", "overall_sbs"]].to_string(index=False))

print("\n── Models in toxicity_scores.csv ──")
tdf = pd.read_csv(os.path.join(OUTPUT_DIR, "toxicity_scores.csv"))
print(tdf[["model", "language", "stbs_toxicity"]].to_string(index=False))

## STEP 5 — Generate all charts
Reads from outputs/stereotype_scores.csv and outputs/toxicity_scores.csv
Saves all charts to outputs/charts/


In [ ]:

chart_paths = generate_all_charts()

print("\n✓ All charts generated:")
for name, path in chart_paths.items():
    print(f"  {name}: {os.path.basename(path)}")


In [ ]:
print("\n" + "="*70)
print(" FULL RESULTS SUMMARY")
print("="*70)

stereo_df_out  = pd.read_csv(os.path.join(OUTPUT_DIR, "stereotype_scores.csv"))
toxicity_df_out = pd.read_csv(os.path.join(OUTPUT_DIR, "toxicity_scores.csv"))

# ── Stereotype: overall SBS ───────────────────────────────────────────────────
print("\n── OVERALL STEREOTYPE BIAS SCORE (SBS %) ──")
pivot_sbs = stereo_df_out.pivot(index="model", columns="language", values="overall_sbs")
print(pivot_sbs.to_string())

# ── Stereotype: per-category averages across models ───────────────────────────
cat_cols = [c for c in stereo_df_out.columns if c.startswith("csbs_")]
print(f"\n── CATEGORY-LEVEL SBS (%) — Cross-model averages ──")
for lang in LANGUAGES:
    lang_df = stereo_df_out[stereo_df_out["language"] == lang]
    print(f"\n  {lang}:")
    for col in cat_cols:
        cat_name = col.replace("csbs_", "")
        avg = lang_df[col].mean()
        print(f"    {cat_name:30s}: {avg:.2f}%")

# ── Toxicity: STBS per subtype ────────────────────────────────────────────────
sub_cols = [c for c in toxicity_df_out.columns if c.startswith("stbs_")]
print(f"\n── SUBTYPE-LEVEL TBS — Cross-model averages ──")
for lang in LANGUAGES:
    lang_df = toxicity_df_out[toxicity_df_out["language"] == lang]
    print(f"\n  {lang}:")
    for col in sub_cols:
        sub_name = col.replace("stbs_", "").replace("_", " ").title()
        avg = lang_df[col].mean()
        print(f"    {sub_name:30s}: {avg:.6f}")

print("\n" + "="*70)
print(" Pipeline complete. All outputs saved to:", OUTPUT_DIR)
print("="*70)


## Optional — Run single model or single language (for debugging/testing)
Uncomment to test just one model on English before running the full pipeline:


In [ ]:

# stereo_scores_test, _ = run_stereotype_evaluation(
#     model_names=["BLOOM-560M"],
#     languages=["English"],
# )
# print(stereo_scores_test)

# tox_scores_test, _ = run_toxicity_evaluation(
#     model_names=["BLOOM-560M"],
#     languages=["English"],
# )
# print(tox_scores_test)
